# Machine Learning Lab – Practical 6: Clustering on Sales Data

## Problem Statement
Implement **K-Means clustering and Hierarchical clustering** on the `sales_data_sample.csv` dataset. Determine a suitable number of clusters using the **Elbow Method**.

**Dataset source:** Sample Sales Data (Kaggle / SPPU Machine Learning dataset repository)  
**Dataset file:** `sales_data_sample.csv`

## Aim
To group similar sales records using unsupervised machine-learning algorithms and understand the clusters formed.

## Learning outcomes
After completing this practical, you should be able to:
1. Explain supervised and unsupervised learning.
2. Understand K-Means and Hierarchical clustering.
3. Select numeric features and standardize them.
4. Use the Elbow Method to estimate the number of clusters.
5. Fit K-Means and Hierarchical clustering models.
6. Visualize and interpret cluster assignments.

**Reference to your uploaded `Notebook-6.ipynb`:** The main workflow is retained: load the sales dataset, select numerical columns, standardize them, apply the Elbow Method, fit K-Means with an example of 3 clusters, visualize SALES against PRICEEACH, and apply Ward-linkage Hierarchical clustering with 3 clusters. Additional checks and explanations are included to make the practical more reliable and easier to understand.

## 1. Theory: What is Clustering?

Clustering is an **unsupervised machine-learning technique** used to divide data points into groups called clusters.

Records in the same cluster should be similar to each other, while records in different clusters should be relatively dissimilar.

Unlike classification:
- Clustering does not require a target/output label in the training data.
- The algorithm discovers groups from patterns in the input features.
- The cluster numbers (0, 1, 2, ...) are identifiers, not meaningful class names by themselves.

### Example
A business may group sales transactions into clusters based on sales amount, quantity ordered, and product price. These groups may help identify different sales patterns.

## 2. Theory: K-Means Clustering

K-Means is an unsupervised algorithm that divides data into a user-specified number of clusters, `k`.

### Working steps
1. Choose the number of clusters `k`.
2. Initialize `k` cluster centroids.
3. Calculate the distance between each data point and each centroid.
4. Assign each point to its nearest centroid.
5. Recalculate each centroid as the mean of the points assigned to it.
6. Repeat assignment and centroid updates until the clusters stop changing significantly or the iteration limit is reached.

### Objective function
K-Means attempts to minimize the Within-Cluster Sum of Squares (WCSS), also called inertia:

\[
WCSS = \sum_{j=1}^{k}\sum_{x_i\in C_j} ||x_i-\mu_j||^2
\]

where:
- `k` = number of clusters.
- `Cj` = set of points in cluster `j`.
- `xi` = a data point.
- `μj` = centroid (mean) of cluster `j`.

### Advantages
- Simple and relatively fast.
- Works well when clusters are reasonably compact and separated.
- Easy to interpret using centroids and cluster counts.

### Limitations
- The number of clusters must be chosen in advance.
- Sensitive to feature scaling and outliers.
- Different initial centroid positions can produce different results.
- It may not work well for irregularly shaped clusters.

## 3. Theory: Elbow Method

The Elbow Method helps estimate a suitable value of `k`.

1. Fit K-Means for a range of cluster counts, commonly `k = 1` to `10`.
2. Record the inertia (WCSS) for each value of `k`.
3. Plot the number of clusters against inertia.
4. Look for the point where the decrease in inertia begins to slow noticeably. This bend is called the **elbow**.

Adding more clusters always tends to reduce inertia, so the goal is not simply to choose the largest `k`. The elbow is a practical visual guide, not a guaranteed mathematically optimal answer.

In the uploaded practical, the final K-Means model uses `k = 3` as an example. Inspect your own elbow plot and change `chosen_k` if your instructor or plot indicates another value.

## 4. Theory: Hierarchical Clustering

Hierarchical clustering builds a hierarchy of clusters. The uploaded practical uses **Agglomerative Hierarchical clustering**, which follows a bottom-up approach:

1. Initially, each data point is treated as its own cluster.
2. Calculate distances between clusters.
3. Merge the two closest clusters.
4. Repeat until all points are merged into one hierarchy.
5. Use a cut through the hierarchy to obtain the required number of clusters.

### Ward linkage
The code uses `method='ward'`. Ward's method merges clusters in a way that aims to minimize the increase in within-cluster variance.

### Dendrogram
A dendrogram is a tree diagram showing how clusters are merged. The height of a merge represents the distance / dissimilarity at which clusters are joined. Cutting the dendrogram at an appropriate height gives a set of clusters.

### K-Means vs Hierarchical clustering

| K-Means | Hierarchical |
|---|---|
| Requires `k` before fitting | Builds a hierarchy; clusters can be extracted afterward |
| Uses centroids | Uses distances and linkage |
| Reassigns points iteratively | Merges clusters in agglomerative form |
| Usually efficient on larger datasets | Can be memory/time intensive for large datasets |
| Output includes cluster centroids | Output can be visualized using a dendrogram |

## 5. Import Required Libraries

- `pandas`: loads and manipulates the CSV data.
- `matplotlib.pyplot`: creates plots.
- `seaborn`: creates an easy-to-read scatter plot.
- `StandardScaler`: standardizes numeric features.
- `KMeans`: implements K-Means clustering.
- `linkage`, `dendrogram`, `fcluster`: perform and visualize hierarchical clustering.
- `numpy`: handles numeric values and missing/infinite-value checks.

In [ ]:
# 1. Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

## 6. Load the Dataset

`pd.read_csv()` loads the CSV file into a DataFrame.

The uploaded code uses `encoding='latin1'` and `on_bad_lines='skip'`. These options are retained:
- `encoding='latin1'` helps read text containing characters outside standard ASCII.
- `on_bad_lines='skip'` skips malformed CSV lines.

Keep `sales_data_sample.csv` in the same folder as this notebook, or update the path below.

In [ ]:
# 2. Load Dataset
df = pd.read_csv(
    "sales_data_sample.csv",
    encoding="latin1",
    on_bad_lines="skip"
)

print("Dataset shape:", df.shape)
display(df.head())

### Understand the dataset

Before clustering, inspect:
- the number of rows and columns,
- column names and data types,
- missing values,
- numeric summary statistics.

This is important because clustering uses numeric measurements and distance calculations.

In [ ]:
# Dataset inspection
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nNumeric summary:")
display(df.describe())

## 7. Select Numerical Features

The uploaded practical selects every numeric column using:

`df.select_dtypes(include=['int64', 'float64'])`

This is preserved below. Numeric identifiers such as order numbers can sometimes be poor clustering features because their numeric magnitude may not represent meaningful similarity. For this practical, the default follows your uploaded code; you can instead select meaningful columns such as `SALES`, `QUANTITYORDERED`, and `PRICEEACH` if your instructor asks for that feature set.

The code also removes columns that are entirely missing and replaces infinite values with missing values. Missing numeric values are filled with the median so the scaler and clustering algorithms can run.

In [ ]:
# 3. Select Numerical Features for Clustering
X = df.select_dtypes(include=["int64", "float64"]).copy()

# Convert positive/negative infinity to missing values
X = X.replace([np.inf, -np.inf], np.nan)

# Remove numeric columns that contain no usable values
X = X.dropna(axis=1, how="all")

if X.shape[1] == 0:
    raise ValueError("No usable numeric columns were found in the dataset.")

# Fill missing numeric values with each column's median
X = X.fillna(X.median())

print("Numeric features used for clustering:")
print(X.columns.tolist())

print("\nFeature preview:")
display(X.head())

print("\nFeature matrix shape:", X.shape)

### Optional: Select only business-relevant columns

The original notebook includes this commented example:

`SALES`, `QUANTITYORDERED`, and `PRICEEACH`.

If you want to use only those three features, run the following cell instead of the all-numeric selection cell above. It checks that the required columns exist. Do not run both feature-selection cells without deciding which feature set your practical should use.

In [ ]:
# OPTIONAL FEATURE SELECTION
# Uncomment and run this cell instead of the previous feature-selection cell
# if you want to use only these three features.

# selected_columns = ["SALES", "QUANTITYORDERED", "PRICEEACH"]
# missing_columns = [col for col in selected_columns if col not in df.columns]
# if missing_columns:
#     raise ValueError(f"Missing required columns: {missing_columns}")
# X = df[selected_columns].copy()
# X = X.replace([np.inf, -np.inf], np.nan)
# X = X.fillna(X.median())

## 8. Standardize the Data

Distance-based algorithms can be dominated by features with larger numeric scales. `StandardScaler` transforms each feature to have approximately:
- mean = 0
- standard deviation = 1

Formula:

\[
z = \frac{x-\mu}{\sigma}
\]

where `x` is the original value, `μ` is the feature mean, and `σ` is the feature standard deviation.

The clustering algorithms below use `X_scaled`, the standardized feature matrix.

In [ ]:
# 4. Standardize Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)
print("\nFirst five standardized rows:")
print(X_scaled[:5])

## 9. K-Means Clustering – Elbow Method

The code fits K-Means for each `k` from 1 through 10 and stores `inertia_` (WCSS).

`n_init=10` runs K-Means with multiple centroid initializations and keeps the best result. It also avoids a common warning about the default initialization count in newer scikit-learn versions.

The elbow plot helps decide the number of clusters.

In [ ]:
# 5. K-Means: Determine k using the Elbow Method
wcss = []
k_values = range(1, 11)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_scaled)
    wcss.append(model.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), wcss, marker="o")
plt.title("Elbow Method for Optimal Number of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia / WCSS")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

### Choose the number of clusters

Look at the elbow curve generated above. The uploaded notebook uses 3 clusters as an example. The next cell therefore keeps `chosen_k = 3`.

If your elbow plot clearly suggests a different value, change `chosen_k` before running the next cell. It must be between 2 and 10 for this practical's tested range.

In [ ]:
# Select the number of clusters based on the Elbow plot
chosen_k = 3  # Change this value if your Elbow plot suggests another k

if not 2 <= chosen_k <= 10:
    raise ValueError("chosen_k should be between 2 and 10.")

# Train final K-Means model
kmeans = KMeans(
    n_clusters=chosen_k,
    random_state=42,
    n_init=10
)

clusters_kmeans = kmeans.fit_predict(X_scaled)

# Store cluster labels in the original DataFrame
df["KMeans_Cluster"] = clusters_kmeans

print("K-Means model fitted with k =", chosen_k)
print("\nCluster counts:")
print(df["KMeans_Cluster"].value_counts().sort_index())

print("\nCluster centers in standardized feature space:")
print(kmeans.cluster_centers_)

### Visualize K-Means clusters

The original notebook plots `SALES` on the x-axis and `PRICEEACH` on the y-axis, with cluster membership shown using color.

This is a two-dimensional view of the clusters. If the model uses more than two features, the visualization still shows only the relationship between these two selected columns; the clustering itself was performed using all selected features.

In [ ]:
# K-Means scatter plot using SALES and PRICEEACH
required_plot_columns = ["SALES", "PRICEEACH"]

if all(col in df.columns for col in required_plot_columns):
    plt.figure(figsize=(9, 6))
    sns.scatterplot(
        x="SALES",
        y="PRICEEACH",
        hue="KMeans_Cluster",
        data=df,
        palette="viridis",
        s=55
    )
    plt.title("K-Means Clustering of Sales Data")
    plt.xlabel("Sales")
    plt.ylabel("Price Each")
    plt.legend(title="K-Means Cluster")
    plt.show()
else:
    print(
        "Scatter plot skipped: the dataset must contain "
        "'SALES' and 'PRICEEACH' columns."
    )

## 10. Hierarchical Clustering

The uploaded practical uses SciPy's `linkage()` with Ward's method.

- `linkage(X_scaled, method='ward')` builds the hierarchical merge structure.
- `dendrogram()` displays the hierarchy.
- `fcluster()` extracts a chosen number of clusters from the hierarchy.

A full dendrogram can become crowded when there are many rows. The plot below uses `truncate_mode='lastp'` to show the final 30 merged groups, making it easier to read. The linkage matrix is still calculated from the complete standardized dataset.

In [ ]:
# 6. Hierarchical Clustering
# Create linkage matrix using Ward's method
Z = linkage(X_scaled, method="ward")

# Plot a readable, truncated dendrogram
plt.figure(figsize=(12, 6))
dendrogram(
    Z,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True
)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)")
plt.xlabel("Merged groups / sample counts")
plt.ylabel("Ward distance")
plt.tight_layout()
plt.show()

### Assign hierarchical cluster labels

The original practical uses `fcluster(Z, t=3, criterion='maxclust')` to extract three clusters. We retain that approach and use the same `chosen_k` value as K-Means so the two results can be compared at a high level.

Note: cluster IDs from K-Means and Hierarchical clustering are arbitrary. For example, cluster 0 from K-Means is not automatically equivalent to cluster 1 from Hierarchical clustering.

In [ ]:
# Extract clusters from the hierarchical tree
clusters_hier = fcluster(
    Z,
    t=chosen_k,
    criterion="maxclust"
)

df["Hierarchical_Cluster"] = clusters_hier

print("Hierarchical clustering counts:")
print(pd.Series(clusters_hier).value_counts().sort_index())

## 11. Compare the Two Clustering Results

This comparison displays the number of records assigned to each cluster by each method. The cluster labels are arbitrary, so compare overall grouping patterns and cluster sizes rather than assuming that cluster numbers match across algorithms.

In [ ]:
# Compare cluster sizes
kmeans_counts = df["KMeans_Cluster"].value_counts().sort_index()
hier_counts = df["Hierarchical_Cluster"].value_counts().sort_index()

print("K-Means cluster sizes:")
print(kmeans_counts)

print("\nHierarchical cluster sizes:")
print(hier_counts)

comparison = pd.DataFrame({
    "KMeans_Count": kmeans_counts,
    "Hierarchical_Count": hier_counts
}).fillna(0).astype(int)

print("\nCluster-size comparison (labels are method-specific):")
display(comparison)

## 12. Optional: Describe the K-Means Clusters

A useful way to understand clusters is to calculate the average original feature values within each K-Means cluster. This gives a simple profile of the groups.

The values below are in the original feature units, not standardized units.

In [ ]:
# Average numeric feature values for each K-Means cluster
cluster_profile = df.groupby("KMeans_Cluster")[X.columns].mean().round(2)

print("Average feature values by K-Means cluster:")
display(cluster_profile)

## 13. Important Concepts for Viva

- **Unsupervised learning:** learning patterns from data without a supplied target label.
- **Cluster:** a group of similar data points.
- **Centroid:** the mean position of points in a K-Means cluster.
- **WCSS / inertia:** sum of squared distances between points and their assigned centroids.
- **Elbow point:** the bend in the WCSS curve used as a guide for choosing `k`.
- **Standardization:** scaling features to comparable ranges using mean and standard deviation.
- **Linkage:** the rule used to calculate distance between clusters during hierarchical clustering.
- **Ward method:** a linkage method that minimizes the increase in within-cluster variance at each merge.
- **Dendrogram:** a tree diagram showing the hierarchical merging process.

## 14. Common Errors and Troubleshooting

| Error / symptom | Possible reason | What to do |
|---|---|---|
| `FileNotFoundError` | CSV is not in the notebook's working directory | Put `sales_data_sample.csv` beside the notebook or update the path |
| `KeyError: 'SALES'` or `'PRICEEACH'` | Expected columns are absent or named differently | Check `df.columns`; use the actual column names |
| `ValueError: Input X contains NaN` | Missing values remain in selected features | Run the numeric cleaning cell and inspect missing-value counts |
| K-Means warning about `n_init` | scikit-learn version expects an explicit initialization count | This notebook sets `n_init=10` |
| Dendrogram is too crowded | Too many individual leaves are displayed | The notebook uses a truncated dendrogram |
| Memory error during linkage | Hierarchical clustering can require substantial memory for large datasets | Use a representative sample for dendrogram exploration or a smaller feature/sample set, and disclose the sampling |
| Clusters look unexpected | Numeric identifiers, outliers, or irrelevant features may influence distance | Inspect selected columns and consider the three meaningful sales features |

**Important:** The original notebook selected all numeric columns. Numeric identifiers such as `ORDERNUMBER` may influence the clusters even though their magnitude may not represent business similarity. If this causes poor results, use the optional three-feature selection cell and clearly mention the selected features in your practical record.

## 15. Conclusion

In this practical, clustering is performed on `sales_data_sample.csv` using two unsupervised learning algorithms: K-Means and Hierarchical clustering. Numerical features are selected and standardized before calculating distances. The Elbow Method is used to estimate a suitable number of K-Means clusters. K-Means is then fitted, and its groups are visualized using `SALES` and `PRICEEACH`. Hierarchical clustering is performed using Ward linkage, and a dendrogram is plotted to show the cluster-merging structure. Finally, cluster counts and average feature profiles are inspected.

**Complete after execution:**  
- Number of clusters selected from the Elbow plot: ______  
- K-Means cluster sizes: ______  
- Hierarchical cluster sizes: ______  
- Main observation about the groups: ______

## 16. Viva Questions and Answers

**Q1. What is clustering?**  
Clustering is an unsupervised learning technique that groups similar data points together.

**Q2. What is the aim of this practical?**  
To apply K-Means and Hierarchical clustering to sales data and estimate the number of clusters using the Elbow Method.

**Q3. What is unsupervised learning?**  
A type of machine learning in which the data does not contain a supplied target label for the model to learn.

**Q4. Which dataset is used?**  
`sales_data_sample.csv`.

**Q5. What is K-Means?**  
An algorithm that partitions data into `k` clusters by assigning points to their nearest centroid.

**Q6. What does K represent?**  
The number of clusters to form.

**Q7. What is a centroid?**  
The mean position of all data points assigned to a particular K-Means cluster.

**Q8. What is WCSS?**  
Within-Cluster Sum of Squares; it measures the sum of squared distances from points to their assigned cluster centroids.

**Q9. What is inertia in scikit-learn K-Means?**  
It is the K-Means within-cluster sum of squared distances, also known as WCSS.

**Q10. What is the Elbow Method?**  
A method that plots WCSS against different values of `k` and looks for a bend where the improvement begins to slow.

**Q11. Why does WCSS decrease when K increases?**  
More clusters allow points to be closer to their assigned centroids, so within-cluster variation generally decreases.

**Q12. Why not always choose a very large K?**  
It creates many small groups and can overcomplicate the interpretation; the aim is a useful balance between fit and simplicity.

**Q13. Why is StandardScaler used?**  
To put numeric features on comparable scales because clustering algorithms use distances.

**Q14. What is the StandardScaler formula?**  
`z = (x - mean) / standard deviation`.

**Q15. What is `random_state=42` used for?**  
It makes the random initialization reproducible.

**Q16. What is `n_init=10`?**  
K-Means is run with 10 different centroid initializations, and the best result is retained.

**Q17. What is Hierarchical clustering?**  
A clustering method that creates a hierarchy of nested clusters.

**Q18. What is Agglomerative clustering?**  
A bottom-up hierarchical method that starts with individual points and repeatedly merges clusters.

**Q19. What is a dendrogram?**  
A tree diagram that shows the order and distance at which clusters are merged.

**Q20. What is linkage?**  
A rule for measuring the distance between clusters.

**Q21. Which linkage method is used in this practical?**  
Ward linkage.

**Q22. What is special about Ward linkage?**  
It chooses merges that minimize the increase in within-cluster variance.

**Q23. What does `fcluster()` do?**  
It extracts flat cluster labels from a hierarchical linkage matrix.

**Q24. What does `criterion='maxclust'` mean?**  
It asks for no more than the specified maximum number of clusters.

**Q25. What is the difference between K-Means and Hierarchical clustering?**  
K-Means partitions data around centroids for a chosen `k`; Hierarchical clustering builds a tree of merges that can later be cut into clusters.

**Q26. Why are cluster labels arbitrary?**  
The numbers identify groups only; the number itself does not represent a ranking or class meaning.

**Q27. Why can outliers affect K-Means?**  
Centroids are means, and extreme values can pull the means away from the majority of points.

**Q28. Why should numeric identifiers be treated carefully?**  
Their numeric values may not represent meaningful similarity, but distance-based algorithms can still use them.

**Q29. Does clustering require a target column?**  
No. Clustering is unsupervised and groups records using input features.

**Q30. What is the purpose of the scatter plot in this practical?**  
To display the cluster assignments in two selected dimensions, `SALES` and `PRICEEACH`.

**Q31. If the model uses many features, does the scatter plot show all of them?**  
No. It shows only the two selected axes; the clustering may have used additional features.

**Q32. What is one limitation of Hierarchical clustering?**  
It can require substantial memory and computation for large datasets.

**Q33. Can K-Means determine the best number of clusters automatically?**  
No. The user chooses `k`; the Elbow Method is one way to estimate a suitable value.

**Q34. Why is data standardization important before clustering?**  
It prevents features with large numeric ranges from dominating distance calculations.

## 17. Final Practical Checklist

- [ ] `sales_data_sample.csv` is in the correct folder.
- [ ] Dataset loaded and inspected.
- [ ] Numeric features selected.
- [ ] Missing and infinite numeric values checked.
- [ ] Features standardized using `StandardScaler`.
- [ ] Elbow curve generated for `k = 1` to `10`.
- [ ] Chosen cluster count recorded and justified from the plot.
- [ ] K-Means model fitted and cluster labels added.
- [ ] SALES vs PRICEEACH scatter plot generated (if columns exist).
- [ ] Ward linkage matrix generated.
- [ ] Dendrogram plotted.
- [ ] Hierarchical cluster labels generated.
- [ ] Cluster counts and profiles interpreted.
- [ ] Conclusion completed using actual results.

**Execution note:** This notebook has been syntax-checked, but the clustering results have not been generated here because the dataset file was not included with the notebook upload. Run all cells with `sales_data_sample.csv` available in the same folder.